Bibliotecas

In [0]:
import requests
import pandas
import json
from pyspark.sql.functions import current_timestamp, lit

Dados Bronze `API` de Estatisticas de PIX

In [0]:
"""
API_URL = "https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?@DataBase='202506'&$format=json&$select=AnoMes,Municipio_Ibge,Municipio,Estado_Ibge,Estado,Sigla_Regiao,Regiao,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,VL_RecebedorPF,QT_RecebedorPF,VL_RecebedorPJ,QT_RecebedorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ,QT_PES_RecebedorPF,QT_PES_RecebedorPJ"

data = get_data(API_URL)

#df_pandas to spark dataframe
df_spark = spark.createDataFrame([(json.dumps(data),)], ["raw_json"]) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source", lit("pix_api_statistics"))
    
df_spark.write.mode("append").saveAsTable("project_apis.bronze.pix_statistics")
"""

In [0]:
API_URL = "https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?@DataBase='202506'&$format=json&$select=AnoMes,Municipio_Ibge,Municipio,Estado_Ibge,Estado,Sigla_Regiao,Regiao,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,VL_RecebedorPF,QT_RecebedorPF,VL_RecebedorPJ,QT_RecebedorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ,QT_PES_RecebedorPF,QT_PES_RecebedorPJ"

response = requests.get(API_URL)
print(response.json())

In [0]:
def get_data(url):
    response = requests.get(url)
    return response.json()

def parse_json(response):
    itens = response['value']
    charlist = []
    for item in itens:
        char = json.dumps(item)
        charlist.append(char)
    return charlist



#use
data = get_data(API_URL)

final_rows = parse_json(data)



In [0]:
df_spark = spark.createDataFrame([(row,) for row in final_rows], ["raw_json"]) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source", lit("pix_api_statistics"))

display(df_spark)

Recriando Bronze


In [0]:
%sql
DROP TABLE IF EXISTS project_apis.bronze.pix_statistics

In [0]:
df_spark.write.mode("append").saveAsTable("project_apis.bronze.pix_statistics")